<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/precision_recall_example.png" align="center" width="30%">
</div>

<br>

# PRECISION, RECALL, AND THRESHOLD

<br>

**About:** A hands-on introduction to precision, recall, and decision thresholds for binary classification, demonstrated using a breast cancer diagnostic dataset and logistic regression.

**Learning Goals:**
- Define precision and recall and explain the trade-off between them
- Apply decision threshold adjustment to shift a model's precision-recall balance
- Build and tune a logistic regression pipeline using stratified cross-validation
- Interpret model coefficients as a crude feature importance signal

**Keywords:** precision, recall, threshold, logistic regression, binary classification, confusion matrix, scikit-learn

**Prerequisite Knowledge:** (1) Python basics, (2) pandas and NumPy, (3) introductory supervised learning concepts

**Target User:** Self-learners who understand what a classifier does and want to go deeper on how to evaluate and tune one for imbalanced-cost tasks like medical diagnosis

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: PRECISION AND RECALL - THE CONCEPTS](#Part_1)
> #### [PART 2: DATASET AND EXPLORATION](#Part_2)
> #### [PART 3: DATA PREPARATION](#Part_3)
> #### [PART 4: MODEL TRAINING AND PRECISION-RECALL ANALYSIS](#Part_4)

<br>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **PRECISION** AND **RECALL** - The Concepts

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/precision_recall_example.png" align="center" width="40%" padding="10"><br>
    <br>
    Precision-recall diagram illustrating the trade-off between the two metrics
</div>

#### CONTENTS:

> [PART 1.1: THE CONFUSION MATRIX](#Part_1_1)<br>
> [PART 1.2: PRECISION AND RECALL DEFINED](#Part_1_2)<br>
> [PART 1.3: THE DECISION THRESHOLD](#Part_1_3)<br>
> [PART 1.4: WHEN TO PRIORITIZE PRECISION VS. RECALL](#Part_1_4)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: THE CONFUSION MATRIX

<br>

Before defining precision and recall, we need a shared vocabulary for the four types of classification outcomes. Every binary classifier produces predictions that fall into one of four buckets:

| Outcome | Meaning |
|---|---|
| **True Positive (TP)** | The model predicted positive, and the true label is positive |
| **True Negative (TN)** | The model predicted negative, and the true label is negative |
| **False Positive (FP)** | The model predicted positive, but the true label is negative (a false alarm) |
| **False Negative (FN)** | The model predicted negative, but the true label is positive (a miss) |

Arranged as a matrix where rows = true labels and columns = predicted labels, this gives the **confusion matrix**:

$$\begin{bmatrix} TN & FP \\ FN & TP \end{bmatrix}$$

where $TN$ is top-left, $TP$ is bottom-right, $FP$ is top-right, and $FN$ is bottom-left.

The confusion matrix is the foundation for every classification metric - precision, recall, F1, specificity, and others all derive from these four counts.

___

**Note:** scikit-learn's `confusion_matrix` returns this matrix in the order `[[TN, FP], [FN, TP]]`, which matches the layout above. Confirm at [scikit-learn docs](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html).

___

In [ ]:
## Confusion matrix from scratch - run to verify layout
y_true_example = np.array([1, 0, 1, 1, 0, 0, 1, 0])
y_pred_example = np.array([1, 0, 0, 1, 1, 0, 1, 0])

C = confusion_matrix(y_true_example, y_pred_example)
print('Confusion matrix (rows=true label, cols=predicted label):')
print(C)
print()
print('TN:', C[0, 0], '| FP:', C[0, 1])
print('FN:', C[1, 0], '| TP:', C[1, 1])

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: PRECISION AND RECALL DEFINED

<br>

**Precision** answers the question: *of all the samples the model labeled positive, what fraction are actually positive?*

$$\text{Precision} = \frac{TP}{TP + FP}$$

where $TP$ is the count of true positives and $FP$ is the count of false positives. A model with high precision rarely cries wolf - when it flags something as positive, it is usually right.

**Recall** (also called *sensitivity* or *true positive rate*) answers a different question: *of all the samples that are actually positive, what fraction did the model catch?*

$$\text{Recall} = \frac{TP}{TP + FN}$$

where $FN$ is the count of false negatives. A model with high recall rarely misses a real positive - it catches most of them, even if it also raises some false alarms.

___

**Note:** These definitions are stable mathematical identities and hold across all implementations. Source: [scikit-learn classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics).

___

In [ ]:
## Compute precision and recall from a confusion matrix
def precision_from_cm(C):
    tp, fp = C[1, 1], C[0, 1]
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall_from_cm(C):
    tp, fn = C[1, 1], C[1, 0]
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

C = confusion_matrix(y_true_example, y_pred_example)
print('Precision:', precision_from_cm(C))
print('Recall:   ', recall_from_cm(C))

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: THE DECISION THRESHOLD

<br>

Most classifiers do not output a hard 0 or 1 - they output a **probability score** between 0 and 1. A **decision threshold** $t$ converts that score into a binary prediction: predict positive if the score exceeds $t$, negative otherwise.

By convention, $t = 0.5$ is the default. But the default is not always best:

- **Lower $t$** - the model flags more samples as positive. Recall rises (fewer actual positives are missed), but precision tends to fall (more false alarms).
- **Higher $t$** - the model only flags high-confidence positives. Precision rises (fewer false alarms), but recall falls (more actual positives are missed).

This is the **precision-recall trade-off**: for a fixed model, you cannot simultaneously maximize both metrics. You can shift the balance by moving $t$, but you trade one for the other.

The choice of $t$ is a business or clinical decision, not a statistical one:

- A cancer screening tool is better off with **high recall** and lower precision - missing a true cancer case is far more costly than a false alarm that leads to a follow-up test.
- A spam filter with access to a "bulk" folder might prefer **high precision** - users would rather miss some spam than lose a legitimate email.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.4: WHEN TO PRIORITIZE PRECISION VS. RECALL

<br>

The asymmetry between false positives and false negatives is what drives the choice:

<br>

**Prioritize recall when false negatives are costly**

A missed cancer diagnosis can cost a life. A fraud transaction that slips through costs money. In these cases, set a low threshold - catch as many positives as possible and accept more false alarms.

<br>

**Prioritize precision when false positives are costly**

Wrongly removing a legitimate user's account, incorrectly flagging a safe drug, or sending a marketing email to someone who opted out - these false positives have real consequences. Set a higher threshold so that positive predictions are reliable.

<br>

**When neither cost dominates, use F1**

If false positives and false negatives are roughly equally undesirable, the **F1 score** (the harmonic mean of precision and recall) provides a single balanced metric:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

The harmonic mean penalizes extreme imbalance: a model with 100% recall and 1% precision gets $F_1 \approx 0.02$, not 50.5%.

___

**Sources Consulted:**
- [Google ML Crash Course - Precision and Recall](https://developers.google.com/machine-learning/crash-course/classification/precision-and-recall)
- [scikit-learn: Classification Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics)

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **A model trained to flag fraudulent credit card transactions achieves 99% precision and 20% recall. A product manager says "99% precision is great - we're almost never wrong when we flag something." Write code to compute what percentage of actual fraud cases this model is catching. Then write two sentences explaining why this model would likely be unacceptable in production.**

<br>

```python
# Given counts - do not change these
tp = 200    # transactions correctly flagged as fraud
fp = 2      # legitimate transactions wrongly flagged
fn = 800    # fraud transactions the model missed

### YOUR CODE HERE ###
precision = ...
recall = ...

print(f'Precision: {precision:.1%}')
print(f'Recall:    {recall:.1%}')
```

<hr style="border: 2px solid#003262;" />

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **DATASET** and Exploration

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/breast_cancer_cell_nuclei.png" align="center" width="40%" padding="10"><br>
    <br>
    Magnified breast fine-needle aspirate showing cell nuclei whose features are measured in this dataset
</div>

#### CONTENTS:

> [PART 2.1: DATASET INFORMATION](#Part_2_1)<br>
> [PART 2.2: ATTRIBUTE INFORMATION](#Part_2_2)<br>
> [PART 2.3: SANITY CHECK AND EXPLORATION](#Part_2_3)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: DATASET INFORMATION

<br>

We use the [Breast Cancer Wisconsin (Diagnostic) dataset](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic)) from the UCI Machine Learning Repository. The dataset contains digitized images of fine-needle aspirates (FNA) of breast masses. Each row represents one patient sample.

Key facts:
- **569 samples**, each described by **30 numeric features** plus an ID and a diagnosis label
- **Label**: `M` (malignant) or `B` (benign)
- **No missing values** (verified by the original authors)
- Features are computed from the image of the cell nucleus: radius, texture, perimeter, area, smoothness, and five others, each measured as mean, standard error, and worst (largest) value across nuclei in the image

Breast cancer context: early and accurate detection is critical because treatment outcomes are strongly tied to stage at diagnosis. A model that misses a malignant case (false negative) has a higher real-world cost than one that triggers an unnecessary follow-up biopsy (false positive). This makes recall the primary metric for this use case.

___

**Sources Consulted:**
- [UCI Breast Cancer Wisconsin (Diagnostic) dataset](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic))
- [National Breast Cancer Foundation - Breast Cancer Facts](https://www.nationalbreastcancer.org/breast-cancer-facts)

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: ATTRIBUTE INFORMATION

<br>

Each of the 30 feature columns is a measurement computed from the cell nucleus image. Ten base measurements each appear three times:

| Base measurement | Mean column | Standard error column | Worst column |
|---|---|---|---|
| radius | `radius_mean` | `radius_se` | `radius_worst` |
| texture | `texture_mean` | `texture_se` | `texture_worst` |
| perimeter | `perimeter_mean` | `perimeter_se` | `perimeter_worst` |
| area | `area_mean` | `area_se` | `area_worst` |
| smoothness | `smoothness_mean` | `smoothness_se` | `smoothness_worst` |
| compactness | `compactness_mean` | `compactness_se` | `compactness_worst` |
| concavity | `concavity_mean` | `concavity_se` | `concavity_worst` |
| concave points | `concave points_mean` | `concave points_se` | `concave points_worst` |
| symmetry | `symmetry_mean` | `symmetry_se` | `symmetry_worst` |
| fractal dimension | `fractal_dimension_mean` | `fractal_dimension_se` | `fractal_dimension_worst` |

"Worst" refers to the mean of the three largest values for that measurement across all nuclei in the image - not the single worst value. Malignant tumors tend to have larger, more irregular nuclei, which is reflected in higher values for radius, area, concavity, and related features.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_2_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.3: SANITY CHECK AND EXPLORATION

<br>

We load the dataset and run three checks before touching the data: shape, missing values, and duplicates. Each check catches a different class of problem - an unexpected shape suggests the wrong file was loaded; missing values require imputation or column removal before modeling; duplicates can leak the test set into training if not removed before the split.

In [ ]:
## Load dataset
df = pd.read_csv('BreastCancer.csv')
df.head(5)

In [ ]:
## Data sanity check
print('Dataset shape:', df.shape)
print('\nMissing entries per column:')
print(df.isna().sum(axis=0))
print('\nDuplicated rows:', df.duplicated().sum())

In [ ]:
## Data types - confirm 'diagnosis' is categorical, all others numeric
df.dtypes

In [ ]:
## Promote 'diagnosis' to a pandas category type for efficient grouping
df = df.astype({'diagnosis': 'category'})

## Class distribution
print('Class distribution:')
print(df['diagnosis'].value_counts())
print(f"\nMalignant fraction: {df['diagnosis'].value_counts()['M'] / len(df):.1%}")

The dataset is moderately imbalanced: roughly 37% malignant, 63% benign. This is mild enough that standard cross-validation works without special resampling, but it means accuracy alone would be misleading - a model that always predicts benign would achieve 63% accuracy while catching zero cancer cases.

In [ ]:
## Distribution of radius_mean by diagnosis class
fig, ax = plt.subplots(figsize=(8, 4))
for label, group in df.groupby('diagnosis')['radius_mean']:
    ax.hist(group, bins=20, alpha=0.6, label=label)
ax.set_xlabel('radius_mean')
ax.set_ylabel('Count')
ax.set_title('Distribution of radius_mean by Diagnosis')
ax.legend()
plt.tight_layout()
plt.show()

The radius_mean distributions overlap but are clearly shifted - malignant tumors tend to have larger radii. This separation means the feature carries discriminative signal, though no single feature cleanly separates the classes.

In [ ]:
## Average feature values by diagnosis class (first 10 features)
df.groupby('diagnosis').mean().iloc[:, :10]

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Choose a second feature from the dataset. Plot its distribution split by diagnosis class using the same approach as the `radius_mean` histogram above. Based on the plot, do you expect this feature to be more or less useful than `radius_mean` for predicting malignancy? Explain in one sentence.**

<br>

```python
feature = 'concave points_mean'  # try changing this to another column name

### YOUR CODE HERE ###
```

<hr style="border: 2px solid#003262;" />

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **DATA** Preparation

#### CONTENTS:

> [PART 3.1: INPUT AND LABELS](#Part_3_1)<br>
> [PART 3.2: TRAINING AND TEST SETS](#Part_3_2)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: INPUT AND LABELS

<br>

We separate the feature matrix $X$ from the label vector $y$. The `id` column is excluded because it is an arbitrary patient identifier with no predictive relationship to the diagnosis - including it would give the model a spurious feature that would never generalize.

The `LabelEncoder` maps the string labels `B` and `M` to integers. It assigns labels alphabetically, so `B` becomes 0 (negative class) and `M` becomes 1 (positive class). This ordering matters: scikit-learn's precision, recall, and confusion matrix functions treat class 1 as the positive class by default, which is what we want - malignant is the outcome we care about catching.

In [ ]:
## Define features (X) and labels (y)
X = df.iloc[:, 2:]  # exclude 'id' (col 0) and 'diagnosis' (col 1)
y = df['diagnosis']

# Encode 'B' -> 0, 'M' -> 1 (alphabetical order)
# Verified: LabelEncoder assigns 0 to 'B', 1 to 'M'
y = LabelEncoder().fit_transform(y)

print('Feature matrix shape:', X.shape)
print('Label counts - 0 (Benign):', (y == 0).sum(), '| 1 (Malignant):', (y == 1).sum())

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: TRAINING AND TEST SETS

<br>

We split the data into three sets:
- **Test set (15%)** - held out completely until the very end. Used once to report final model performance on unseen data.
- **Validation set (15% of training)** - used during hyperparameter tuning to choose the best `C` value.
- **Training set (remaining)** - used to fit the model.

The test set is not touched during hyperparameter selection. Doing so would allow the test distribution to leak into model choices, producing an optimistic estimate of generalization performance.

We split training into training+validation via `StratifiedKFold` rather than a simple validation split, because stratification preserves the class ratio in each fold - important when the dataset is moderately imbalanced.

In [ ]:
## Hold out the test set before any scaling or modeling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=0
)

## Split the training set further into a smaller train set and a validation set
## Used for hyperparameter tuning - X_t/y_t are for fitting, X_v/y_v for evaluation
X_t, X_v, y_t, y_v = train_test_split(
    X_train, y_train, test_size=0.15, random_state=0
)

print('Train+Val size:', len(X_train), '| Test size:', len(X_test))
print('Tuning train:', len(X_t), '| Validation:', len(X_v))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Explain in two sentences why we must split the data into train and test sets before any scaling is applied, rather than scaling the entire dataset first and then splitting. What specific problem would arise if we scaled first?**

<br>

```python
# No code required for this question.
# Write your answer as a comment below.

### YOUR ANSWER HERE ###
# ...
```

<hr style="border: 2px solid#003262;" />

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **MODEL TRAINING** and Precision-Recall Analysis

#### CONTENTS:

> [PART 4.1: BUILDING A PIPELINE](#Part_4_1)<br>
> [PART 4.2: HYPERPARAMETER TUNING VIA CROSS-VALIDATION](#Part_4_2)<br>
> [PART 4.3: THE PRECISION-RECALL TRADE-OFF](#Part_4_3)<br>
> [PART 4.4: THRESHOLD MOVING](#Part_4_4)<br>
> [PART 4.5: TESTING THE FINAL MODEL](#Part_4_5)<br>
> [PART 4.6: FEATURE IMPORTANCE](#Part_4_6)<br>

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: BUILDING A PIPELINE

<br>

We combine feature scaling and logistic regression into a single `Pipeline`. The pipeline ensures that the scaler is fit only on training data and that the same transformation is applied consistently to validation and test data during `predict` calls. Without this encapsulation, it is easy to accidentally fit the scaler on the full training set and then re-apply it during cross-validation folds, introducing a subtle form of data leakage.

The `StandardScaler` centers each feature to zero mean and unit variance. Logistic regression uses gradient-based optimization and is sensitive to feature scale - unscaled features with very different magnitudes slow convergence and can produce poorly-calibrated coefficients.

___

**Note:** The `solver` parameter in `LogisticRegression` is version-sensitive. `'newton-cg'` is a valid solver for this use case (small-to-medium datasets, L2 penalty). Verify available solvers at [sklearn LogisticRegression docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for the version you are running. `# TODO: verify solver default has not changed in your sklearn version`

___

In [ ]:
## Build the pipeline: StandardScaler -> LogisticRegression
# TODO: verify solver and default penalty against current sklearn docs before relying on outputs
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(solver='newton-cg', max_iter=1000))
])

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.2: HYPERPARAMETER TUNING VIA CROSS-VALIDATION

<br>

The regularization parameter `C` in logistic regression controls the strength of the L2 penalty applied to the model's coefficients. Smaller `C` means stronger regularization - the model is penalized more for large coefficients, which reduces variance but can increase bias. Larger `C` allows the model to fit more closely to the training data.

We search over a logarithmic range of `C` values using `GridSearchCV` with `StratifiedKFold`. We run this search **twice** - once optimizing for recall, once for precision - to show that the choice of scoring metric directly affects which model is selected.

Why a logarithmic range? Model behavior tends to change in proportion to the order of magnitude of `C`, not linearly. A logarithmic range covers several orders of magnitude efficiently.

In [ ]:
## Tune C via cross-validation, scoring by recall
# We tune for recall first because recall is the primary metric for cancer detection
parameters = {'lr__C': np.logspace(-3, 3, 10)}
split = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)

grid_recall = GridSearchCV(pipeline, parameters, cv=split, scoring='recall')
grid_recall.fit(X_t, y_t)
pipeline_best_recall = grid_recall.best_estimator_

cv_recall = np.mean(cross_val_score(pipeline_best_recall, X_t, y_t, cv=split, scoring='recall'))
cv_prec = np.mean(cross_val_score(pipeline_best_recall, X_t, y_t, cv=split, scoring='precision'))

print('Best pipeline (optimized for recall)')
print(f'  C = {pipeline_best_recall.named_steps["lr"].C:.4f}')
print(f'  CV recall:    {cv_recall:.3f}')
print(f'  CV precision: {cv_prec:.3f}')

In [ ]:
## Tune C via cross-validation, scoring by precision
grid_prec = GridSearchCV(pipeline, parameters, cv=split, scoring='precision')
grid_prec.fit(X_t, y_t)
pipeline_best_precision = grid_prec.best_estimator_

cv_recall_p = np.mean(cross_val_score(pipeline_best_precision, X_t, y_t, cv=split, scoring='recall'))
cv_prec_p = np.mean(cross_val_score(pipeline_best_precision, X_t, y_t, cv=split, scoring='precision'))

print('Best pipeline (optimized for precision)')
print(f'  C = {pipeline_best_precision.named_steps["lr"].C:.4f}')
print(f'  CV recall:    {cv_recall_p:.3f}')
print(f'  CV precision: {cv_prec_p:.3f}')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.3: THE PRECISION-RECALL TRADE-OFF

<br>

The two tuning runs above select different `C` values. This is the precision-recall trade-off made visible: a model that was searched to maximize recall sacrifices some precision compared to one searched to maximize precision - and vice versa.

The key insight is that the trade-off is not just a threshold effect. Even at the same threshold, a model whose hyperparameters were tuned for recall will produce different probability scores than one tuned for precision. The scoring metric used during `GridSearchCV` shapes the model itself, not just how you interpret its output.

In [ ]:
## Summarize the trade-off
print('Metric comparison across both pipelines:')
print(f'{"Metric":<20} {"Recall-optimized":>18} {"Precision-optimized":>20}')
print('-' * 60)
print(f'{"CV Recall":<20} {cv_recall:>18.3f} {cv_recall_p:>20.3f}')
print(f'{"CV Precision":<20} {cv_prec:>18.3f} {cv_prec_p:>20.3f}')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.4: THRESHOLD MOVING

<br>

For the recall-optimized model, we now vary the decision threshold and observe how precision and recall move in opposite directions. This plot is the empirical version of the trade-off described in Part 1.

`predict_proba` returns the model's probability estimate for each class. We use the column at index 1 (probability of being malignant) and compare it to each candidate threshold. Lowering the threshold accepts more uncertain positives, which catches more malignant cases (higher recall) but also flags more benign cases as malignant (lower precision).

In [ ]:
## Fit recall-optimized model on the tuning training set
pipeline_best_recall.fit(X_t, y_t)

## Sweep decision thresholds from 0 to 1
n = 20
thresh = np.linspace(0.01, 0.99, n)
precisions = np.zeros(n)
recalls = np.zeros(n)

for i, t in enumerate(thresh):
    y_pred = (pipeline_best_recall.predict_proba(X_v)[:, 1] >= t).astype(int)
    C = confusion_matrix(y_v, y_pred)
    tp, fp, fn = C[1, 1], C[0, 1], C[1, 0]
    precisions[i] = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recalls[i] = tp / (tp + fn) if (tp + fn) > 0 else 0.0

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh, precisions, 'r-o', markersize=4, label='Precision')
ax.plot(thresh, recalls, 'b-o', markersize=4, label='Recall')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall vs. Decision Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

As expected, precision and recall move in opposite directions as the threshold changes. The clinical decision about where to set the threshold depends on how the model will be used:
- A first-pass screening tool should use a low threshold (catch everything, follow up on positives).
- A tool that triggers an invasive procedure should use a higher threshold (only flag high-confidence cases).

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4_5'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.5: TESTING THE FINAL MODEL

<br>

We now select a threshold and evaluate the recall-optimized model on the held-out test set. The test set has not been used for any fitting or hyperparameter decisions - this is its first use, and it gives an estimate of how the model will perform on genuinely new data.

We choose `thresh_chosen = 0.4` as a threshold that (based on the validation set curve above) provides high recall while keeping precision at an acceptable level for a screening context. In a real deployment, this choice would be made in collaboration with clinical staff.

In [ ]:
## Retrain on the full training set (train + validation) before testing
pipeline_best_recall.fit(X_train, y_train)

thresh_chosen = 0.4
y_pred_test = (pipeline_best_recall.predict_proba(X_test)[:, 1] >= thresh_chosen).astype(int)
C_test = confusion_matrix(y_test, y_pred_test)

tp, fp, fn, tn = C_test[1, 1], C_test[0, 1], C_test[1, 0], C_test[0, 0]
test_recall = tp / (tp + fn)
test_precision = tp / (tp + fp)

print('Test set confusion matrix:')
print(C_test)
print(f'\nTest recall:    {test_recall:.3f}')
print(f'Test precision: {test_precision:.3f}')
print(f'\nFalse negatives (missed cancers): {fn}')
print(f'False positives (unnecessary alarms): {fp}')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4_6'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.6: FEATURE IMPORTANCE

<br>

Logistic regression coefficients serve as a crude feature importance signal. A large positive coefficient means the feature pushes the model toward predicting malignant (class 1); a large negative coefficient pushes toward benign (class 0). "Crude" because the coefficients are on the scaled feature space - two features with similar coefficients but different natural scales would have different effects in the original data.

This analysis can surface which measurements the model relies on most heavily, which is useful for domain experts to validate or question the model's reasoning.

___

**Note:** Coefficient magnitudes are only comparable across features if those features have been scaled, which our pipeline ensures. Comparing coefficients from an unscaled logistic regression would be misleading.

___

In [ ]:
## Feature importance from logistic regression coefficients
pipeline_best_recall.fit(X_t, y_t)  # refit on tuning set for consistent reference
coef = pipeline_best_recall.named_steps['lr'].coef_[0]
feature_names = X.columns

importance_df = pd.DataFrame({'feature': feature_names, 'coefficient': coef})
importance_df = importance_df.reindex(importance_df['coefficient'].abs().sort_values(ascending=False).index)

print('Top 10 features by absolute coefficient:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
## Compare mean values of top features by diagnosis class
top_features = importance_df['feature'].head(3).tolist()
for feat in top_features:
    print(df[['diagnosis', feat]].groupby('diagnosis').mean())
    print()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Using the threshold-precision-recall curve from Part 4.4, find the decision threshold at which recall first drops below 0.90. Report the precision at that threshold. Then explain in one sentence whether you would accept this model at that threshold for a cancer screening context, and why.**

<br>

```python
## Reuse the thresh, precisions, recalls arrays from Part 4.4

### YOUR CODE HERE ###
# Find threshold where recall first drops below 0.90
threshold_found = ...
precision_at_threshold = ...

print(f'Threshold: {threshold_found:.2f}')
print(f'Precision: {precision_at_threshold:.3f}')
```

<hr style="border: 2px solid#003262;" />

<hr style="border: 6px solid#003262;" />